In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "FD_MODEL_cache/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
SOURCE_REGIONS = [  # regions for "foundation model"
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['SJM', 'CENTRALASIA']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'SJM': False,
    'CENTRALASIA': False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in SOURCE_REGIONS + TARGET_REGIONS
}

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Model assets:

#### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["data_monthly"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses

glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

In [ ]:
assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

print(f"Foundation model assets: {len(assets_full['ds_train'])} sequences")

#### Test regions:

In [ ]:
RECOMPUTE_SPLITS = False
save_path = BASE_DIR / "glacier_splits_FT.json"

if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        output = json.load(f)
    splits = {
        region: {
            "pool_glaciers": data["pool_glaciers"],
            "holdout_glaciers": data["holdout_glaciers"],
            "actual_pool_frac": data["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": data["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": data["sinkhorn_holdout_vs_region"],
            "blur_joint": data["blur_joint"],
            "best_seed": data["best_seed"],
            "best_score": data["best_score"],
        }
        for region, data in output.items()
    }
    print(f"Loaded splits from: {save_path}")
    for region, split in splits.items():
        print(f"\n{region}:")
        print(f"  Pool glaciers    : {split['pool_glaciers']}")
        print(f"  Holdout glaciers : {split['holdout_glaciers']}")
        print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
        print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

else:
    # remap cache keys to what build_global_scalers_multi_source_simple expects
    res_source_only = {
        region: {
            "df_train": res_xreg_by_source[region]["data_monthly"],
            "df_test": res_xreg_by_source[region]["data_monthly_aug"],
        }
        for region in SOURCE_REGIONS
    }

    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_source_only,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )
    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_source_only,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    splits = {}
    for region in TARGET_REGIONS:
        print(f"\n{'='*50}")
        print(f"Splitting region: {region}")

        # use data_monthly (not aug) as the pool/holdout split input
        df_target = res_xreg_by_source[region]["data_monthly"]

        split = split_pool_holdout_sinkhorn(
            df_region=df_target,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            pool_frac=0.2,
            n_restarts=50,
            seed=cfg.seed,
        )
        splits[region] = split

        print(f"  Pool glaciers    : {split['pool_glaciers']}")
        print(f"  Holdout glaciers : {split['holdout_glaciers']}")
        print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
        print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")
        print(f"  Sk(pool↔region)  : {split['sinkhorn_pool_vs_region']:.4f}")
        print(
            f"  Sk(holdout↔reg)  : {split['sinkhorn_holdout_vs_region']:.4f}")

    output = {
        region: {
            "pool_glaciers": split["pool_glaciers"],
            "holdout_glaciers": split["holdout_glaciers"],
            "actual_pool_frac": split["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": split["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": split["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": split["sinkhorn_holdout_vs_region"],
            "blur_joint": split["blur_joint"],
            "best_seed": split["best_seed"],
            "best_score": split["best_score"],
        }
        for region, split in splits.items()
    }
    with open(save_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nAll splits saved to: {save_path}")

In [ ]:
FT_FRAC = 0.10
FORCE_RECOMPUTE = False

ft_assets = {}

for region in TARGET_REGIONS:
    print(f"\n{'='*50}")
    print(f"Building FT assets for: {region}")

    df_monthly = res_xreg_by_source[region]["data_monthly"]
    df_monthly_aug = res_xreg_by_source[region]["data_monthly_aug"]
    head_pad = res_xreg_by_source[region]["months_head_pad"]
    tail_pad = res_xreg_by_source[region]["months_tail_pad"]

    if region == "SJM":
        # --- ID-level split (too few glaciers) ---
        winter_ids = df_monthly[df_monthly["PERIOD"] ==
                                "winter"]["ID"].unique().tolist()
        annual_ids = df_monthly[df_monthly["PERIOD"] ==
                                "annual"]["ID"].unique().tolist()

        rng = np.random.default_rng(cfg.seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        n_ft_w = max(1, int(len(winter_ids) * FT_FRAC))
        n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))

        ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
        test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

        print(f"  FT  : {n_ft_w} winter + {n_ft_a} annual = {len(ft_ids)} IDs")
        print(f"  Test: {len(winter_ids)-n_ft_w} winter + "
              f"{len(annual_ids)-n_ft_a} annual = {len(test_ids)} IDs")

        df_ft_loss = df_monthly[df_monthly["ID"].isin(ft_ids)]
        df_ft_full = df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)]
        df_test_loss = df_monthly[df_monthly["ID"].isin(test_ids)]
        df_test_full = df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)]

    else:
        # --- Glacier-level split (Sinkhorn optimization) ---
        pool_glaciers = splits[region]["pool_glaciers"]
        holdout_glaciers = splits[region]["holdout_glaciers"]

        print(f"  Pool    : {len(pool_glaciers)} glaciers")
        print(f"  Holdout : {len(holdout_glaciers)} glaciers")

        df_ft_loss = df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)]
        df_ft_full = df_monthly_aug[df_monthly_aug["GLACIER"].isin(
            pool_glaciers)]
        df_test_loss = df_monthly[df_monthly["GLACIER"].isin(holdout_glaciers)]
        df_test_full = df_monthly_aug[df_monthly_aug["GLACIER"].isin(
            holdout_glaciers)]

    exp_key = f"FD_to_{region}"

    ds_ft = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"{exp_key}_FT",
        df_loss=df_ft_loss,
        df_full=df_ft_full,
        months_head_pad=head_pad,
        months_tail_pad=tail_pad,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=FORCE_RECOMPUTE,
        kind="ft",
        show_progress=False,
    )

    ds_test = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"{exp_key}_TEST",
        df_loss=df_test_loss,
        df_full=df_test_full,
        months_head_pad=head_pad,
        months_tail_pad=tail_pad,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=FORCE_RECOMPUTE,
        kind="test",
        show_progress=False,
    )

    ft_train_idx, ft_val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_ft), val_ratio=0.2, seed=cfg.seed)

    ft_assets[region] = {
        "ds_ft": ds_ft,
        "ds_test": ds_test,
        "ft_train_idx": ft_train_idx,
        "ft_val_idx": ft_val_idx,
    }

    print(f"  ds_ft  : {len(ds_ft)} sequences "
          f"({ds_ft.iw.sum().item()} winter | {ds_ft.ia.sum().item()} annual)")
    print(
        f"  ds_test: {len(ds_test)} sequences "
        f"({ds_test.iw.sum().item()} winter | {ds_test.ia.sum().item()} annual)"
    )

## Transformer:

In [ ]:
best_params_gs = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 2,
    'static_hidden': 64,
    'static_dropout': 0.0,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

### Pooled model:

In [ ]:
TRAIN_FLAG = True
MODEL_DATE = "2026-05-28"
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_foundation, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=best_params_gs,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=True,
    epochs=150,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Foundation model saved to: {model_path}")

In [ ]:
print(f"\n{'='*60}")
print("Zero-shot evaluation of foundation model on target regions")
print(f"{'='*60}")

for region in TARGET_REGIONS:
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_full["ds_train"]
    )
    ds_scaler.make_loaders(
        train_idx=assets_full["train_idx"],
        val_idx=assets_full["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets[region]["ds_test"]
    )
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy,
        ds_scaler,
        batch_size=128,
        seed=cfg.seed,
    )
    metrics, _ = model_foundation.evaluate_with_preds(device, test_dl, ds_test_copy)
    print(f"\n  {region} (zero-shot):")
    print(f"    RMSE_a={metrics['RMSE_annual']:.3f}  "
          f"RMSE_w={metrics['RMSE_winter']:.3f}  "
          f"R2_a={metrics['R2_annual']:.3f}  "
          f"R2_w={metrics['R2_winter']:.3f}")

In [ ]:
plot_configs = [
    (f"{region} (zero-shot)", model_foundation, assets_full)
    for region in TARGET_REGIONS
]

for region in TARGET_REGIONS:
    plot_pred_vs_truth_grid(
        plot_configs=[(f"{region} (zero-shot)", model_foundation, assets_full)],
        ds_test=ft_assets[region]["ds_test"],
        device=device,
        cfg=cfg,
        ncols=1,
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        title=f"Foundation model → {region} (zero-shot)",
        save_path=BASE_DIR / f"figures/foundation_zeroshot_{region}",
        figsize=(5, 5),
    )

### Fine tuning:

In [ ]:
TRAIN_FLAG = True
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

# scaler donor: always cloned from the foundation training assets
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"]
)
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

ft_models = {}

for region in TARGET_REGIONS:
    ft_models[region] = {}
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region}")

    for strategy in ["head_only", "adapter", "full"]:
        ft_models[region][strategy] = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_foundation,
            ds_ft=ft_assets[region]["ds_ft"],
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=best_params_gs,
            model_filename=str(
                models_dir_ft / f"transformer_ft_{strategy}_{region}_{MODEL_DATE}.pt"
            ),
            strategy=strategy,
            epochs=50,
            lr_factor=0.1,
            force_retrain=TRAIN_FLAG,
        )
        print(f"\n  [{region}] {strategy} done")

In [ ]:
print(f"\n{'='*60}")
print("Fine-tuning results vs zero-shot baseline")
print(f"{'='*60}")

for region in TARGET_REGIONS:
    print(f"\n--- {region} ---")

    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets[region]["ds_test"]
    )
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy,
        ds_pooled_scaler,
        batch_size=128,
        seed=cfg.seed,
    )

    # zero-shot baseline
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets[region]["ds_test"]
    )
    m, _ = model_foundation.evaluate_with_preds(device, test_dl, ds_test_copy)
    print(f"  {'no_ft':25s}  RMSE_a={m['RMSE_annual']:.3f}  "
          f"RMSE_w={m['RMSE_winter']:.3f}  R2_a={m['R2_annual']:.3f}  "
          f"R2_w={m['R2_winter']:.3f}")

    # fine-tuned strategies
    for strategy, model_ft in ft_models[region].items():
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ft_assets[region]["ds_test"]
        )
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy,
            ds_pooled_scaler,
            batch_size=128,
            seed=cfg.seed,
        )
        m, _ = model_ft.evaluate_with_preds(device, test_dl, ds_test_copy)
        print(f"  {strategy:25s}  RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

In [ ]:
for region in TARGET_REGIONS:
    plot_configs = [
        (f"no_ft (zero-shot)", model_foundation, assets_full),
        *[
            (strategy, ft_models[region][strategy], assets_full)
            for strategy in ["head_only", "adapter", "full"]
        ],
    ]

    plot_pred_vs_truth_grid(
        plot_configs=plot_configs,
        ds_test=ft_assets[region]["ds_test"],
        device=device,
        cfg=cfg,
        ncols=4,
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        title=f"Foundation model → {region}: zero-shot vs fine-tuning strategies",
        save_path=BASE_DIR / f"figures/foundation_ft_strategies_{region}",
        figsize=(20, 5),
    )